# PRUEBA CON INTENSIDADES DESDE EL 15-07-2025 AL 15-05-2026

In [2]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
import xgboost as xgb
from sklearn.linear_model import Ridge
import joblib
import warnings
warnings.filterwarnings('ignore')

print("====================================================================")
print("     MÓDULO DE REPLAY EN TIEMPO REAL: INFERENCIA CONTINUA MOE       ")
print("====================================================================\n")

# 1. Cargar el dataset de intensidades limpio de duplicados
df_stream = pd.read_csv('data/processed/intensidades_clean_no_duplicates.csv')
df_stream['timestamp'] = pd.to_datetime(df_stream['timestamp'])
df_stream = df_stream.sort_values('timestamp').reset_index(drop=True)

print(f"[DataOps] Registros únicos listos para simulación en vivo: {len(df_stream)}")

# 2. Configurar y ajustar la Gating Network (GMM) definitiva de producción
# Se asumen los coeficientes de ortogonalización calculados analíticamente previos
alfa_fe, beta_fe = 0.5, 100  
alfa_cu, beta_cu = 0.3, 50
alfa_zn, beta_zn = 0.01, 5
alfa_mo, beta_mo = 0.05, 12

df_stream['n1fe_ortho'] = df_stream['n1fe'] - (alfa_fe * df_stream['n6sc'] + beta_fe)
df_stream['n2cu_ortho'] = df_stream['n2cu'] - (alfa_cu * df_stream['n6sc'] + beta_cu)
df_stream['n3zn_ortho'] = df_stream['n3zn'] - (alfa_zn * df_stream['n6sc'] + beta_zn)
df_stream['n4mo_ortho'] = df_stream['n4mo'] - (alfa_mo * df_stream['n6sc'] + beta_mo)

features_ortho = ['n1fe_ortho', 'n2cu_ortho', 'n3zn_ortho', 'n4mo_ortho']
X_ortho = df_stream[features_ortho].values

print("[Gating] Calibrando Gaussian Mixture Model (Tied) con el espectro histórico unificado...")
gmm_gating = GaussianMixture(n_components=3, covariance_type='tied', random_state=42)
gmm_gating.fit(X_ortho)
print("[OK] Gating Network optimizada para producción.\n")

# 3. Cargar los Expertos de Regresión Supervisados (.pkl guardado previamente)
moe_artifacts = joblib.load('models/moe_soft_sensor_architecture.pkl')
moe_experts = moe_artifacts['models']
scaler_c1 = moe_artifacts['scaler_c1']

# 4. Iniciar bucle de simulación del pipeline en producción (Streaming Simulator)
results_history = []

print("--- Procesando flujo continuo (Simulación de tags OPC UA) ---")
for idx, row in df_stream.iterrows():
    t_stamp = row['timestamp']
    x_sample = X_ortho[idx].reshape(1, -1)
    
    # Paso A: Obtener densidades probabilísticas blandas del GMM
    cluster_weights = gmm_gating.predict_proba(x_sample)[0]
    W0, W1, W2 = cluster_weights[0], cluster_weights[1], cluster_weights[2]
    
    # Paso B: Ejecutar inferencias en paralelo de los expertos locales
    # Experto Clúster 0 (Régimen Estándar)
    y_pred_c0 = {
        'pCu': moe_experts[0]['pCu'].predict(x_sample)[0],
        'pFe': moe_experts[0]['pFe'].predict(x_sample)[0],
        'pMo': moe_experts[0]['pMo'].predict(x_sample)[0],
        'pZn': moe_experts[0]['pZn'].predict(x_sample)[0]
    }
    
    # Experto Clúster 1 (Régimen de Alta Ley - requiere escalador para SVR Mo)
    x_sample_scaled = scaler_c1.transform(x_sample)
    y_pred_c1 = {
        'pCu': moe_experts[1]['pCu'].predict(x_sample)[0],
        'pFe': moe_experts[1]['pFe'].predict(x_sample)[0],
        'pMo': moe_experts[1]['pMo'].predict(x_sample_scaled)[0], # SVR experto
        'pZn': moe_experts[1]['pZn'].predict(x_sample)[0]
    }
    
    # Experto Clúster 2 (Régimen Complejo - Medias estáticas robustas)
    y_pred_c2 = {
        'pCu': moe_experts[2]['pCu'],
        'pFe': moe_experts[2]['pFe'],
        'pMo': moe_experts[2]['pMo'],
        'pZn': moe_experts[2]['pZn']
    }
    
    # Paso C: Fusión de salida por combinación lineal probabilística (Fórmula MoE)
    final_pred = {}
    for element in ['pCu', 'pFe', 'pMo', 'pZn']:
        final_pred[element] = (W0 * y_pred_c0[element]) + (W1 * y_pred_c1[element]) + (W2 * y_pred_c2[element])
        
    # Guardar estado en el log histórico del soft-sensor
    results_history.append({
        'timestamp': t_stamp,
        'W0_Standard': W0,
        'W1_HighGrade': W1,
        'W2_Complex': W2,
        'pCu_Est': final_pred['pCu'],
        'pFe_Est': final_pred['pFe'],
        'pMo_Est': final_pred['pMo'],
        'pZn_Est': final_pred['pZn']
    })
    
    # Imprimir un log de control en intervalos para auditoría de la consola
    if idx % 100 == 0 or idx == len(df_stream) - 1:
        print(f"[{t_stamp}] Gating -> W0:{W0:.2f} W1:{W1:.2f} W2:{W2:.2f} | Est %Cu: {final_pred['pCu']:.2f}% | %Mo: {final_pred['pMo']:.2f}%")

# 5. Exportar el dataset de series de tiempo continuas estimadas
df_final_sensor = pd.DataFrame(results_history)
df_final_sensor.to_csv('data/final/soft_sensor_continuous_predictions.csv', index=False)

print("\n====================================================================")
print("     [OK] SIMULACIÓN COMPLETADA Y ARCHIVO MAESTRO EXPORTADO        ")
print("====================================================================")
print("Archivo generado: 'soft_sensor_continuous_predictions.csv'")
print(f"Total de estimaciones continuas inyectadas: {len(df_final_sensor)}")

     MÓDULO DE REPLAY EN TIEMPO REAL: INFERENCIA CONTINUA MOE       

[DataOps] Registros únicos listos para simulación en vivo: 18124
[Gating] Calibrando Gaussian Mixture Model (Tied) con el espectro histórico unificado...
[OK] Gating Network optimizada para producción.

--- Procesando flujo continuo (Simulación de tags OPC UA) ---
[2025-07-15 00:00:00] Gating -> W0:0.44 W1:0.56 W2:0.00 | Est %Cu: 26.36% | %Mo: 2.30%
[2025-07-17 00:24:00] Gating -> W0:0.09 W1:0.88 W2:0.03 | Est %Cu: 27.63% | %Mo: 1.91%
[2025-07-18 15:18:00] Gating -> W0:0.21 W1:0.79 W2:0.00 | Est %Cu: 27.32% | %Mo: 2.02%
[2025-07-20 07:38:00] Gating -> W0:0.52 W1:0.48 W2:0.00 | Est %Cu: 26.04% | %Mo: 2.38%
[2025-07-25 21:11:00] Gating -> W0:0.43 W1:0.57 W2:0.00 | Est %Cu: 26.41% | %Mo: 2.29%
[2025-07-27 12:05:00] Gating -> W0:0.21 W1:0.79 W2:0.00 | Est %Cu: 27.32% | %Mo: 2.06%
[2025-07-29 05:06:00] Gating -> W0:0.35 W1:0.65 W2:0.00 | Est %Cu: 26.76% | %Mo: 2.20%
[2025-07-30 12:29:00] Gating -> W0:0.84 W1:0.16 W2:0.00 